In [22]:
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.animation as animation
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from IPython.display import display
import ipywidgets as widgets
import pandas as pd
import sympy as sp
import scipy.stats as stats
import sympy as sp
import jax
import jax
import jax.numpy as jnp
from jax.scipy.special import ndtr
import cvxpy as cp
import os


%matplotlib ipympl




# Portfolio Optimization via CVaR Minimization

## In the Classical Markowitz Framework

Our risk definition was based on variance. We had two extreme points:

- **Minimum Variance Portfolio**: find the weights that result in the smallest variance
- **Maximum Return Portfolio**: maximize return without considering risk (variance)

The points between those two extremes give us the **Efficient Frontier**.

## Changing the Risk Definition: Variance to CVaR

Now we change the risk definition from **Variance** to **CVaR**:

$$
\min_w \text{CVaR}_\alpha(w) \quad \text{s.t.} \quad \mu^\top w \geq \mu_{target}
$$

This formulation lets us answer questions like:

- "I want at least 0.1% daily return; minimize my risk." (Formulation 1, the return constraint is the natural framing)
- "I'm willing to accept at most 3% CVaR; give me the best possible return for that." (Dual Formulatiom, the risk constraint is the natural framing, the "risk appetite" framing)

## Why Is VaR Not Used as a Risk Definition?

VaR is a poor target for optimization for two reasons:

1. **Non-convex.** As portfolio weights change, the VaR quantile can jump discontinuously, so the optimizer has no reliable downhill direction to follow.
2. **Non-smooth.** VaR is a quantile function; it has no useful gradient with respect to the weights $w$.

CVaR (the expected loss beyond the VaR threshold) fixes both problems: it is convex in $w$ and admits a tractable linear formulation.


# Real Implementation: Rockafellar-Uryasev Reformulation

## The Core Problem

CVaR is defined as "the average loss in the worst $(1-\alpha)$ fraction of scenarios." To compute that directly, you would need to:

1. Compute the loss $-r_t^\top w$ for every scenario $t$ (negative return)
2. Sort all $T$ losses
3. Average the worst $(1-\alpha)T$ of them

Sorting is inherently non-smooth and non-differentiable, making it poorly suited for use within an optimization framework. Rockafellar and Uryasev (2000) developed an approach that achieves the same result without requiring any sorting step."

## The Auxiliary Function $F_\alpha(w,\zeta)$

$$
F_\alpha(w, \zeta) = \zeta + \frac{1}{(1-\alpha)T}\sum_{t=1}^{T} \left[-r_t^\top w - \zeta\right]^+
$$

**Where:**

- $\zeta$ (zeta) is a new variable we introduce ourselves. Think of it as the optimizer's "guess" for what VaR$_\alpha$ is.
- $r_t$ is the return vector for scenario $t$, so $-r_t^\top w$ is the portfolio's **loss** in that scenario.
- $[\cdot]^+ = \max(\cdot, 0)$, the "positive part." It returns the value itself if positive, and 0 otherwise.
- $T$ is the total number of scenarios.

**Intuition:**

For each scenario, compare its loss to the guess $\zeta$:

- If the loss is **below** $\zeta$ (a normal day, not in the tail), the term $[-r_t^\top w - \zeta]^+$ is 0. It contributes nothing.
- If the loss is **above** $\zeta$ (a tail event), **the term equals the EXCESS LOSS beyond $\zeta $ (OUR GUESS FOR START OF TAIL, VAR).**

Summing these excesses and dividing by $(1-\alpha)T$ gives the **AVERAGE EXCESS-over-$\zeta$**, scaled to match the size of the tail. Adding $\zeta$ back on top shifts this from "average excess beyond $\zeta$" to "average loss in the tail," which is exactly CVaR.

What Rockafellar and Uryasev proved is this: if you let the optimizer choose **both** $w$ and $\zeta$ to minimize $F_\alpha(w,\zeta)$, the optimal $\zeta$ automatically lands exactly on VaR$_\alpha$, and the optimal $w$ is the same $w$ you would get by minimizing CVaR directly. You get CVaR minimization "for free," just by adding one extra variable, and $F_\alpha$ is **convex** in $(w,\zeta)$, so it is well-behaved for solvers.


**Numerical example:**

Consider $T = 5$ scenarios with the following portfolio losses:

$$
L = (1,\ 2,\ 3,\ 8,\ 10)
$$

Let $\alpha = 0.6$, so $(1-\alpha)T = 2$. This means the "tail" consists of the two largest losses.

**Step 1  Try $\zeta = 3$:**

| $t$ | $L_t$ | $L_t - \zeta$ | $[L_t - \zeta]^+$ |
|---|---|---|---|
| 1 | 1  | $-2$ | 0 |
| 2 | 2  | $-1$ | 0 |
| 3 | 3  | $0$  | 0 |
| 4 | 8  | $5$  | 5 |
| 5 | 10 | $7$  | 7 |

Sum of excesses $= 12$.

$$
F_\alpha(\zeta=3) = \zeta + \frac{1}{(1-\alpha)T}\sum_t [L_t-\zeta]^+ = 3 + \frac{12}{2} = 9
$$

**Step 2 — Confirm this is the minimum:**

- $\zeta = 2$: excesses are $1, 6, 8$ (sum $=15$) → $F = 2 + 15/2 = 9.5$
- $\zeta = 5$: excesses are $3, 5$ (sum $=8$) → $F = 5 + 8/2 = 9$
- $\zeta = 8$: excess is $2$ (sum $=2$) → $F = 8 + 2/2 = 9$

$F_\alpha(\zeta)$ is flat at $9$ for any $\zeta \in [3,8]$, and increases outside this range.

**Conclusion:**

- $\text{VaR}_{0.6} = 3$ (the smallest $\zeta$ achieving the minimum)
- $\text{CVaR}_{0.6} = 9$

As a check, $\text{CVaR}_{0.6}$ equals the average of the two tail losses directly: $(8+10)/2 = 9$. This confirms that minimizing $F_\alpha(\zeta)$ reproduces the direct definition of CVaR — exactly as Rockafellar and Uryasev's result predicts.




# CVaR Optimization — Notation and Formulation

## Notation

| Symbol                  | Meaning                                                                        |
|-------------------------|--------------------------------------------------------------------------------|
| $x = (x_1, \dots, x_n)$ | decision vector (e.g., portfolio weights)                                      |
| $X$                     | a convex set of feasible decisions                                             |
| $y = (y_1, \dots, y_n)$ | random vector                                                                  |
| $y^j$                   | scenario of random vector $y$, $j = 1, \dots, J$                               |
| $f(x, y)$ = -R          | **loss function** we used -Return throughout the notebooks                     |
| $\alpha$                | confidence level (e.g. 0.95)                                                   |
| $\zeta$                 | auxiliary variable — the optimizer's candidate value for $\mathrm{VaR}_\alpha$ |
| $z_j$                   | auxiliary variable — "excess loss" of scenario $j$ beyond $\zeta$              |
| $\nu$                   | constant, $\nu = \big((1-\alpha)J\big)^{-1}$                                   |




### The Auxiliary Function $F_\alpha(x,\zeta)$

$$
F_\alpha(x, \zeta) = \zeta + \nu \sum_{j=1}^{J} \left[f(x,y^j) - \zeta\right]^+,
\qquad \nu = \big((1-\alpha)J\big)^{-1}
$$


### From $F_\alpha$ to the LP: Where Do the $z_j$ Come From?

The term $[f(x,y^j) - \zeta]^+$ is non-smooth (it's a $\max$ with 0), and a solver cannot work directly with $\max(\cdot,0)$ inside an objective. The standard trick is **epigraph linearization**: introduce one auxiliary variable $z_j$ per scenario, and replace $[f(x,y^j)-\zeta]^+$ with $z_j$, subject to:

$$
z_j \geq f(x,y^j) - \zeta, \qquad z_j \geq 0, \qquad j = 1, \dots, J
$$


These two inequalities together say $z_j \geq \max\big(f(x,y^j)-\zeta,\ 0\big) = [f(x,y^j)-\zeta]^+$, i.e. $z_j$ is an *upper bound* on the excess loss of scenario $j$.

Since $z_j$ only appears in the objective with a **positive** coefficient ($\nu > 0$) that we are **minimizing**, the solver always wants $z_j$ as small as possible. Given the two constraints, the smallest feasible value of $z_j$ is exactly $\max(f(x,y^j)-\zeta, 0)$:

- If $f(x,y^j) - \zeta \leq 0$: the binding constraint is $z_j \geq 0$, so the optimizer pushes $z_j \to 0$.
- If $f(x,y^j) - \zeta > 0$: the binding constraint is $z_j \geq f(x,y^j)-\zeta$, so the optimizer pushes $z_j$ down to exactly $f(x,y^j)-\zeta$.

So at the optimum, $z_j^* = [f(x,y^j)-\zeta^*]^+$ automatically  **without ever writing a $\max$**. This is the entire trick: $z_j$ is the LP's stand-in for "excess tail loss of scenario $j$," and the constraints + minimization force it to take the correct value on their own.

### The Resulting LP

$$
\min_{x \in X} \ \mathrm{CVaR}_\alpha
$$

reduces to the linear programming (LP) problem

$$
\min_{\{x \in X,\ \zeta \in \mathbb{R},\ z \in \mathbb{R}^J\}} \quad \zeta + \nu \sum_{j=1}^{J} z_j
$$

**subject to:**

$$
z_j \geq f(x,y^j) - \zeta, \qquad z_j \geq 0, \qquad j = 1, \dots, J
$$

$$
\nu = \big((1-\alpha)J\big)^{-1} = \text{const}
$$

By solving this LP we find:

- an optimal portfolio $x^*$,
- the corresponding $\mathrm{VaR}$, which equals the lowest optimal $\zeta^*$,
- the minimal $\mathrm{CVaR}$, which equals the optimal value of the linear objective function $\zeta^* + \nu\sum_j z_j^*$.

**Constraints** $x \in X$ may account for various trading constraints, including a mean return constraint (e.g., expected return should exceed 10%).

Similar to return–variance analysis, an **efficient frontier** can be constructed and a **tangent portfolio** found.

---

## Risk Management with CVaR Constraints

### The Idea

Sometimes we don't want to *minimize* CVaR as the objective, instead, we want to keep $\mathrm{CVaR} \leq C$ as a **constraint** while optimizing something else (e.g. maximizing return). The same $(\zeta, z_j)$ trick used in the objective above can be reused inside a constraint.

Recall that at the optimum, $\zeta + \nu \sum_j z_j$ equals exactly $\mathrm{CVaR}_\alpha$ (this is what the LP objective computed). So the constraint

$$
\mathrm{CVaR} \leq C
$$

can be replaced by writing out *that same expression* and bounding it by $C$, together with the same defining constraints on $z_j$:

$$
\zeta + \nu \sum_{j=1}^{J} z_j \leq C
$$

$$
z_j \geq f(x,y^j) - \zeta, \qquad z_j \geq 0, \qquad j = 1, \dots, J
$$

$$
\nu = \big((1-\alpha)J\big)^{-1} = \text{const}
$$




# Testing the Auxiliary Function


In [23]:
%%capture
%run var_cvar.ipynb


In [24]:
historical_portfolio_returns = returns @ weights

historical_portfolio_loss = -historical_portfolio_returns
historical_portfolio_loss

Date
2020-01-03    0.009103
2020-01-06   -0.003621
2020-01-07    0.009393
2020-01-08   -0.002663
2020-01-09   -0.012691
                ...   
2024-12-24   -0.009059
2024-12-26   -0.000788
2024-12-27    0.005839
2024-12-30    0.008605
2024-12-31   -0.003848
Length: 1257, dtype: float64

In [25]:
zeta_guess_of_var = 0.02 #randomly selected, normally optimizer will select
confidence_level = 0.95


In [26]:
auxiliary_function_intermediate_result = pd.DataFrame({
    "historical_portfolio_loss": historical_portfolio_loss,
    "loss_minus_zeta": historical_portfolio_loss - zeta_guess_of_var,
    "positive_loss_minus_zeta-excess_over_zeta": (historical_portfolio_loss - zeta_guess_of_var).clip(lower=0)
})
auxiliary_function_intermediate_result




,historical_portfolio_loss,loss_minus_zeta,positive_loss_minus_zeta-excess_over_zeta
Date,,,
2020-01-03,0.009103,-0.010897,0.0
2020-01-06,-0.003621,-0.023621,0.0
2020-01-07,0.009393,-0.010607,0.0
2020-01-08,-0.002663,-0.022663,0.0
2020-01-09,-0.012691,-0.032691,0.0
...,...,...,...
2024-12-24,-0.009059,-0.029059,0.0
2024-12-26,-0.000788,-0.020788,0.0
2024-12-27,0.005839,-0.014161,0.0


In [27]:
sum_of_excess = auxiliary_function_intermediate_result['positive_loss_minus_zeta-excess_over_zeta'].sum()
print(f"sum_of_excess= {sum_of_excess}")

T_lengt_of_scenarios = len(historical_portfolio_loss)
print(f"length of scenarios= {T_lengt_of_scenarios}")

auxiliary_function_result_CVAR= zeta_guess_of_var + sum_of_excess * 1/ ((1-confidence_level) * T_lengt_of_scenarios)
print(f"auxiliary_function_result_CVAR= {auxiliary_function_result_CVAR}")

sum_of_excess= 0.8642399006124751
length of scenarios= 1257
auxiliary_function_result_CVAR= 0.033750833740850826


### Turning into a widget to explore different values

In [28]:
result_label = widgets.HTML()
display(result_label)

@widgets.interact(
    zeta_interactive=widgets.FloatSlider(
        value=zeta_guess_of_var,
        min=0,
        max=1,
        step=0.001,
        description="ζ (VaR guess)"
    ),
    confidence_level_interactive=widgets.FloatText(
        value=0.95,
        description="α"
    )
)
def try_different_zetas(zeta_interactive, confidence_level_interactive):

    excess = (historical_portfolio_loss - zeta_interactive).clip(lower=0)
    T = len(historical_portfolio_loss)

    F_alpha = zeta_interactive + excess.sum() / ((1 - confidence_level_interactive) * T)
    true_var = np.quantile(historical_portfolio_loss, confidence_level_interactive)

    result_label.value = f"""
    <b>F_α(w, ζ), CVAR based onζ:</b> {F_alpha:.4f}<br>
    <b>Historical CVAR:</b> {historical_CVAR:.4f}<br>

    <b>Historical VaR:</b> {true_var:.4f}<br>
    <b>ζ distance from VaR:</b> {abs(zeta_interactive - true_var):.4f}

    """

HTML(value='')

interactive(children=(FloatSlider(value=0.02, description='ζ (VaR guess)', max=1.0, step=0.001), FloatText(val…

# Minimum CVAR Portfolio Optimization


In [47]:
T_lengt_of_scenarios = len(losses)
confidence_level = 0.95

weights_min_cvar_portfolio = cp.Variable(4)

zeta_guess_of_var = cp.Variable(1)

z_j_loss_minus_zeta = cp.Variable(T_lengt_of_scenarios)
v = 1/ ((1-confidence_level) * T_lengt_of_scenarios)

portfolio_losses_daily_f_x_y = losses.values @ weights_min_cvar_portfolio




In [48]:


portfolio_cvar_with_RU = zeta_guess_of_var + v * cp.sum(z_j_loss_minus_zeta)

constraints = [

    z_j_loss_minus_zeta >= portfolio_losses_daily_f_x_y - zeta_guess_of_var,
    z_j_loss_minus_zeta >= 0,
    cp.sum(weights_min_cvar_portfolio) == 1,
    weights_min_cvar_portfolio >= 0

]

problem = cp.Problem(cp.Minimize(portfolio_cvar_with_RU), constraints)




In [49]:
problem.solve()

0.03057841648515399

In [32]:
weights_min_cvar_portfolio.value

array([0.15803567, 0.05389462, 0.66658407, 0.12148565])

In [33]:
zeta_guess_of_var.value

array([0.01732537])

In [34]:
tickers

['AAPL', 'JPM', 'XOM', 'KO']

In [51]:
portfolio_losses_daily_f_x_y.value

array([ 0.00686102, -0.0019054 ,  0.00777475, ...,  0.00381922,
        0.00781403, -0.00352312], shape=(1257,))

## Commenting the Results

Under equal-weight allocation, the portfolio exhibited a one-day 95% VaR of 1.90%
and a CVaR of 3.36%. Following CVaR minimization via the Rockafellar-Uryasev LP
formulation, the optimizer converged to an XOM-concentrated solution (66% weight),
reducing VaR to 1.70% and CVaR to 3.05%.

The modest improvement suggests that the four assets share relatively similar tail-risk profiles. With all four being large-cap U.S. equities exposed to the same broad market conditions, correlations remain high, leaving the optimizer limited room to meaningfully reduce CVaR through reallocation alone.



# Optimization with Concrete CVAR Target/Threshold
## (for ex, for maximum Return Portfolio)


In [54]:
T_lengt_of_scenarios = len(losses)
confidence_level = 0.95



weights_min_cvar_portfolio = cp.Variable(4)

zeta_guess_of_var = cp.Variable(1)

z_j_loss_minus_zeta = cp.Variable(T_lengt_of_scenarios)
v = 1/ ((1-confidence_level) * T_lengt_of_scenarios)

portfolio_return = returns.values @ weights_min_cvar_portfolio

portfolio_losses_daily_f_x_y = losses.values @ weights_min_cvar_portfolio

portfolio_mean_loss = cp.mean(portfolio_losses_daily_f_x_y)




In [55]:


portfolio_cvar_with_RU = zeta_guess_of_var + v * cp.sum(z_j_loss_minus_zeta)

constraints_max_return = [

    z_j_loss_minus_zeta >= portfolio_losses_daily_f_x_y - zeta_guess_of_var,
    z_j_loss_minus_zeta >= 0,
    cp.sum(weights_min_cvar_portfolio) == 1,
    weights_min_cvar_portfolio >= 0

]







### Maximum Return Portfolio Without CVAR Limitation

using minimize loss for max return

In [56]:
problem = cp.Problem(cp.Minimize(portfolio_mean_loss), constraints_max_return)
problem.solve()

np.float64(-0.001182110396206762)

Comment, we have recieved **negative** LOSS of 0.0011, means Maximum Return Portfolio has a RETURN of %0.11

In [62]:
for i, w in enumerate(weights_min_cvar_portfolio.value):
    print(f"Asset {tickers[i]}: {w:.4f}")

Asset AAPL: 1.0000
Asset JPM: 0.0000
Asset XOM: 0.0000
Asset KO: 0.0000


In [63]:
zeta_guess_of_var.value

array([-0.0011821])

In [67]:
portfolio_cvar_with_RU.value

array([8.55816072])

Both VaR and CVaR values from this problem are meaningless. Since the RU formulation was never part of the objective or constraints, zeta drifted freely during optimization and the resulting VaR/CVaR figures are artifacts of the solver, not meaningful risk estimates.

Calculating historical CVAR with weights:

In [71]:
portfolio_losses =  losses @ weights_min_cvar_portfolio.value
portfolio_losses


Date
2020-01-03    0.009722
2020-01-06   -0.007968
2020-01-07    0.004703
2020-01-08   -0.016087
2020-01-09   -0.021241
                ...   
2024-12-24   -0.011478
2024-12-26   -0.003176
2024-12-27    0.013242
2024-12-30    0.013263
2024-12-31    0.007058
Length: 1257, dtype: float64

In [73]:
VAR = portfolio_losses.quantile(confidence)
VAR

np.float64(0.03012455991991174)

In [75]:
days_which_lies_at_tail = portfolio_losses[portfolio_losses >= VAR ]
historical_CVAR = days_which_lies_at_tail.mean()
historical_CVAR


np.float64(0.04439480001944587)

With the maximum return portfolio (100% AAPL), the historical VaR at the 95% confidence level is 3.01%, and the historical CVaR is 4.44%.

### Maximum Return Portfolio With CVAR Limitation

In [76]:
CVAR_Threshold = 0.035

constraints_max_return_with_threshold = [

    portfolio_cvar_with_RU <= CVAR_Threshold,

    z_j_loss_minus_zeta >= portfolio_losses_daily_f_x_y - zeta_guess_of_var,
    z_j_loss_minus_zeta >= 0,
    cp.sum(weights_min_cvar_portfolio) == 1,
    weights_min_cvar_portfolio >= 0

]


In [77]:
problem = cp.Problem(cp.Minimize(portfolio_mean_loss), constraints_max_return_with_threshold)
problem.solve()

np.float64(-0.0008759329327783556)

In [78]:
weights_min_cvar_portfolio.value

array([0.51340136, 0.16791679, 0.21767126, 0.10101059])

In [80]:
zeta_guess_of_var.value

array([0.02020454])

In [81]:
portfolio_cvar_with_RU.value

array([0.035])

The optimizer was given a CVaR ceiling of 3.5% and tasked with maximizing return subject to that risk constraint.

Compared to the unconstrained max return portfolio, two things changed. First, the return dropped slightly from 0.11% to 0.088%, which is the cost of imposing the risk constraint. Second, the portfolio is now diversified across all four assets instead of being concentrated entirely in AAPL.

AAPL still dominates at 51.3% since it has the highest expected return, but the remaining weight is spread across JPM, XOM, and KO to keep tail risk within bounds.